# Short-Horizon ODE Optimization (A=50)
This notebook demonstrates the mathematically superior approach to finding chaotic game dynamics in high-dimensional continuous systems.

By leveraging **Native Backprop** (`odeint`) paired with **Short-Horizon Optimization (10,000 steps)**, we achieve a 55x optimization speedup while keeping exact, analytically rigorous gradients that avoid chaos-induced explosion.

Once the game is optimized on the short horizon, we evaluate it on a **Long Horizon (400,000 steps)** using the **Discrete OMWU Algorithm** to observe the persistent, structural chaos learned by the optimizer transferring perfectly to reality.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import numpy as np
from src.engine.ode_optimizer import ODEAdjointOptimizer

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


KeyboardInterrupt



## 1. Short-Horizon Optimization (Adam)
We will optimize a 50x50 zero-sum-ish game. We use 10,000 steps to prevent the continuous Lyapunov exponents from exploding the gradient via the Chain Rule.

In [ ]:
# 1. Initialize 50x50 Random Game (Note: Optimizer initializes internally)
A = 2

# 2. Configure Native ODE Optimizer
optimizer = ODEAdjointOptimizer(
    action_sizes=[A, A],
    eta=0.05,
    N_steps=10000,
    projection_mode="tanh",
    lr=0.05,
    device=device,
    objective_type="envelope_trend",
    T1_ratio=0.8
)

print(f"\nStarting optimization! Session ID: {optimizer.session_id}\n")

# 3. Train for 50 epochs
epochs = 50
optimized_payoffs, loss_history, _, _, _ = optimizer.optimize(epochs=epochs)

# Plot Loss
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='blue', linewidth=2)
plt.title("Short-Horizon Optimization Loss (Envelope Trend)")
plt.xlabel("Epoch")
plt.ylabel("Loss (Negative Regret Growth)")
plt.grid(True)
plt.show()

## 2. Long-Horizon Discrete Evaluation
Now that we have successfully shaped the eigenvalues of the game on a short horizon using the smooth ODE surrogate, we evaluate it in the real world. We instantiate the strict discrete `OMWU` algorithm and simulate it for **400,000 steps**.

In [ ]:
from src.games.nplayer_game import NPlayerGame
from src.dynamics.omwu import OMWU
from src.engine.runner import Engine

# Detach and convert optimized payoffs to numpy for discrete engine
payoffs_np = [U.detach().cpu().numpy() for U in optimized_payoffs]

game = NPlayerGame(payoffs_np)
dynamics = OMWU(game, eta=0.05)
engine = Engine(game, dynamics)

# Long Horizon Discrete: 400,000 steps
print("Running 400,000 discrete steps...")
history = engine.run(num_steps=400000)

# Extract Regret Arrays from history
Regret = history['regret']
print("Discrete Simulation Complete!")

## 3. Plotting the Chaos

In [ ]:
# Plot Regret for Player 1 over 400,000 discrete steps
plt.figure(figsize=(15, 6))

# Regret shape is (num_steps, num_players, max_actions)
# We extract player 0 (Player 1) regret
player_0_regret = Regret[:, 0, :A]

for a in range(A):
    plt.plot(player_0_regret[:, a], alpha=0.3)
    
plt.title(f"Player 1 Regret Trajectories over 400,000 Discrete Steps (A={A})")
plt.xlabel("Discrete Step (t)")
plt.ylabel("Regret")
plt.grid(True)
plt.show()